### Statistical analysis (p-value < 0.05)

This notebook shows statistical analysis of our modified models vs baselines.

V_r_a  | (V_r_a (activation function change), V_c1, V_c1_c2, V_m_a_Attn)

EN_m_a | (EN_BN, EN_BN_c1, EN_m_a_Attn)

In [ ]:
### This is for calculating precise scores

import pandas as pd
df_scores = pd.read_excel(r'../../data/raw_data/ROW_DATA_Sonja_add_MV_ECMO_and_ECMO_LPS.xlsx',sheet_name='Total Score',engine='openpyxl')
df_scores_Ct_MV_LP = df_scores[0:19]


this_data = df_scores['PDF/ EXCEL correspodent Data'].values
df_scores

In [ ]:
# reading the scores from image names and make precise_Scores


import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


######################################################################################################################
###Here we add a column to our dataframe to compare different models
first_exp_col =np.zeros(len(df_scores['(sanned slide) Sample Name ']))



tile_based_scores = {}
image_based_scores = {}
slide_based_scores = {}

max_score = max(df_scores['Median'])
min_score = min(df_scores['Median'])

    ##Normalized thresholds
low = (10-(min_score))/(max_score-min_score)
mid = (25-(min_score))/(max_score-min_score)
    
validation_label = np.zeros((8073,), int)

# Reading true labels of validation set
precise_scores = []
image_names = []
with open('short_result/real_scores.txt','r') as f:
    for line in f:
        la = float(line.split()[1])
        if la > mid:
            validation_label = 2
        elif la > low:
            validation_label = 1
        else:
            validation_label = 0
        precise_scores.append(validation_label)
            



np.save('precise_scores.npy',precise_scores)
    

The following cell were run for all models separately

In [15]:
y_pred = []
with open('short_result/V_Models/V_Attn.txt','r') as fout:
    for line in fout:
        y_pred.append(int(line.split()[1]))

np.save('NPY/V_Attn_tile.npy',y_pred)

### This is for extracting slide number from images to add
keys = []
Slide_dict = {}
with open('short_result/V_Models/V_Attn.txt','r') as fout: 
    for line in fout:
        S_num = str(line.split(' ')[0][6:8])
        if '.' in S_num:
            S_num = S_num.replace('.','')
        if '_' in S_num:
            S_num = S_num.replace('_','')
        if S_num not in keys:
            keys.append(S_num)
            Slide_dict[S_num]=[]

fout.close()
### Add all scores of tiles to the slide key
with open('short_result/V_Models/V_Attn.txt','r') as fout: 
    for line in fout:
        key_temp=line.split(' ')[0][6:8]
        if '.' in key_temp:
            key_temp = key_temp.replace('.','')
        if '_' in key_temp:
            key_temp = key_temp.replace('_','')
        
        Slide_dict[key_temp].append(float(line.split(' ')[1].strip()))
#### Change all tile scores to the score of majority of tiles for slide score 
y_pred_slide = np.ones(len(y_pred))  
offset = 0
for k in Slide_dict.keys():
    class_0 = [index for index, element in enumerate(Slide_dict[k]) if element == 0]
    class_1 = [index for index, element in enumerate(Slide_dict[k]) if element == 1]
    class_2 = [index for index, element in enumerate(Slide_dict[k]) if element == 2]
    majority_vote = np.argmax([len(class_0),len(class_1),len(class_2)]) 
    y_pred_slide[offset:offset+len(Slide_dict[k])]= majority_vote
    offset += len(Slide_dict[k])

np.save('NPY/V_Attn_slide.npy',y_pred_slide)


In [2]:
y_pred = []
with open('short_result/E_Models/EN_BN_relu.txt','r') as fout:
    for line in fout:
        y_pred.append(int(line.split()[1]))

np.save('NPY/En_BN_relu_tile.npy',y_pred)

### This is for extracting slide number from images to add
keys = []
Slide_dict = {}
with open('short_result/E_Models/EN_BN_relu.txt','r') as fout: 
    for line in fout:
        S_num = str(line.split(' ')[0][6:8])
        if '.' in S_num:
            S_num = S_num.replace('.','')
        if '_' in S_num:
            S_num = S_num.replace('_','')
        if S_num not in keys:
            keys.append(S_num)
            Slide_dict[S_num]=[]

fout.close()
### Add all scores of tiles to the slide key
with open('short_result/E_Models/EN_BN_relu.txt','r') as fout: 
    for line in fout:
        key_temp=line.split(' ')[0][6:8]
        if '.' in key_temp:
            key_temp = key_temp.replace('.','')
        if '_' in key_temp:
            key_temp = key_temp.replace('_','')
        
        Slide_dict[key_temp].append(float(line.split(' ')[1].strip()))
#### Change all tile scores to the score of majority of tiles for slide score 
y_pred_slide = np.ones(len(y_pred))  
offset = 0
for k in Slide_dict.keys():
    class_0 = [index for index, element in enumerate(Slide_dict[k]) if element == 0]
    class_1 = [index for index, element in enumerate(Slide_dict[k]) if element == 1]
    class_2 = [index for index, element in enumerate(Slide_dict[k]) if element == 2]
    majority_vote = np.argmax([len(class_0),len(class_1),len(class_2)]) 
    y_pred_slide[offset:offset+len(Slide_dict[k])]= majority_vote
    offset += len(Slide_dict[k])

np.save('NPY/EN_BN_relu_slide.npy',y_pred_slide)


After having all predicted labels as an numpy array we will run next cell for comparison.

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import ttest_rel, wilcoxon, friedmanchisquare
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# LOAD .NPY FILES 
# ==========================================

# Load true labels (ground truth)
true_labels = np.load('NPY/precise_scores.npy')  # Shape: (8073,)

# Load predictions for all models
base_pred   = np.load('NPY/V_r_a_tile.npy')      # Shape: (8073,)
model1_pred = np.load('NPY/V_r_a_LR_slide.npy')  # Shape: (8073,)
model2_pred = np.load('NPY/Vgg_c1_slide.npy')  # Shape: (8073,)
model3_pred = np.load('NPY/V_c1_c2_slide.npy')
model4_pred = np.load('NPY/V_Attn_slide.npy')   # Shape: (8073,)




# Stack all predictions
all_predictions = np.column_stack([
    base_pred, model1_pred, model2_pred, 
    model3_pred, model4_pred
])

model_names = ['V_r_a', 'V_r_a(Leaky_ReLU/ReLU)', 'V_c1', 'V_c1_c2', 'V_m_a_Attn']
n_models = all_predictions.shape[1]
n_samples = len(true_labels)

print("="*80)
print(" STATISTICAL SIGNIFICANCE TESTS")
print(f"   Samples: {n_samples}")
print(f"   Models: {n_models} (V_r_a + 4 improved)")
print("="*80)

# ==========================================
# 1. DESCRIPTIVE STATISTICS
# ==========================================

print("\n 1. DESCRIPTIVE STATISTICS")
print("-" * 60)

metrics_df = pd.DataFrame()

for i, name in enumerate(model_names):
    preds = all_predictions[:, i]
    
    # Overall metrics
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, average='weighted', zero_division=0)
    rec = recall_score(true_labels, preds, average='weighted', zero_division=0)
    f1 = f1_score(true_labels, preds, average='weighted', zero_division=0)
    
    # Correctness per sample
    correct = (preds == true_labels).astype(int)
    correct_pct = correct.mean() * 100
    
    metrics_df[name] = [acc, prec, rec, f1, correct_pct]
    print(f"  {name:12s}: Acc={acc:.4f}, F1={f1:.4f}, Correct={correct_pct:.2f}%")

metrics_df.index = ['Accuracy', 'Precision', 'Recall', 'F1', 'Correct (%)']
print("\n", metrics_df.round(4))

# ==========================================
# 2. PER-SAMPLE CORRECTNESS MATRIX
# ==========================================

correctness_matrix = np.zeros((n_samples, n_models))
for i in range(n_models):
    correctness_matrix[:, i] = (all_predictions[:, i] == true_labels).astype(int)

# ==========================================
# 3. PAIRED STATISTICAL TESTS (Each Model vs Base)
# ==========================================

print("\n" + "="*80)
print(" 2. PAIRED STATISTICAL TESTS (Each Model vs Base)")
print("="*80)

results = []

for i in range(1, n_models):
    model_name = model_names[i]
    
    # Get per-sample correctness
    base_correct = correctness_matrix[:, 0]
    model_correct = correctness_matrix[:, i]
    
    # Difference per sample
    diff = model_correct - base_correct

    plt.plot(diff)


    # Descriptive stats
    n_improved = np.sum(diff > 0)
    n_worse = np.sum(diff < 0)
    n_same = np.sum(diff == 0)
    mean_improvement = np.mean(diff) * 100
    
    print(f"\n{'='*60}")
    print(f" {model_name} vs Base (n = {n_samples} samples)")
    print(f"{'='*60}")
    print(f"  Improved: {n_improved} ({n_improved/n_samples*100:.2f}%)")
    print(f"  Worsened: {n_worse} ({n_worse/n_samples*100:.2f}%)")
    print(f"  Same:     {n_same} ({n_same/n_samples*100:.2f}%)")
    print(f"  Mean Improvement: {mean_improvement:.4f}%")
    
    # ----- Test 1: Paired T-Test -----
    t_stat, p_ttest = ttest_rel(base_correct, model_correct)
    
    print(f"\n  PAIRED T-TEST:")
    print(f"    t-statistic: {t_stat:.4f}")
    print(f"    p-value:     {p_ttest:.6f}")
    print(f"    p < 0.05:    {'✅ YES' if p_ttest < 0.05 else '❌ NO'}")
    print(f"    Result:      {'✅ SIGNIFICANT' if p_ttest < 0.05 else '❌ NOT SIGNIFICANT'}")
    
    # ----- Test 2: Wilcoxon Signed-Rank Test -----
    non_zero_diff = diff[diff != 0]
    if len(non_zero_diff) > 0:
        w_stat, p_wilcox = wilcoxon(non_zero_diff)
        print(f"\n  WILCOXON SIGNED-RANK TEST:")
        print(f"    W-statistic: {w_stat:.2f}")
        print(f"    p-value:     {p_wilcox:.6f}")
        print(f"    p < 0.05:    {'✅ YES' if p_wilcox < 0.05 else '❌ NO'}")
        print(f"    Result:      {'✅ SIGNIFICANT' if p_wilcox < 0.05 else '❌ NOT SIGNIFICANT'}")
    else:
        p_wilcox = None
        print(f"\n   WILCOXON TEST: All differences are zero")
    
    # ----- Test 3: McNemar's Test -----
    a = np.sum((base_correct == 1) & (model_correct == 1))
    b = np.sum((base_correct == 1) & (model_correct == 0))
    c = np.sum((base_correct == 0) & (model_correct == 1))
    d = np.sum((base_correct == 0) & (model_correct == 0))
    
    contingency_table = np.array([[a, b], [c, d]])
    mcnemar_result = mcnemar(contingency_table, exact=False, correction=True)
    p_mcnemar = mcnemar_result.pvalue
    
    print(f"\n  McNEMAR'S TEST:")
    print(f"    Contingency table:")
    print(f"                | Base Correct | Base Wrong")
    print(f"    {model_name} Correct |     {a:5d}     |    {c:5d}")
    print(f"    {model_name} Wrong   |     {b:5d}     |    {d:5d}")
    print(f"    p-value: {p_mcnemar:.6f}")
    print(f"    p < 0.05: {'✅ YES' if p_mcnemar < 0.05 else '❌ NO'}")
    print(f"    Result:  {'✅ SIGNIFICANT' if p_mcnemar < 0.05 else '❌ NOT SIGNIFICANT'}")
    
    # ----- Test 4: Bootstrap Confidence Interval -----
    n_bootstrap = 4000
    bootstrap_diffs = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n_samples, size=n_samples, replace=True)
        boot_base = base_correct[idx]
        boot_model = model_correct[idx]
        bootstrap_diffs.append(np.mean(boot_model) - np.mean(boot_base))
    
    ci_lower = np.percentile(bootstrap_diffs, 2.5)
    ci_upper = np.percentile(bootstrap_diffs, 97.5)
    ci_lower_pct = ci_lower * 100
    ci_upper_pct = ci_upper * 100
    contains_zero = ci_lower <= 0 <= ci_upper
    
    print(f"\n  BOOTSTRAP 95% CONFIDENCE INTERVAL:")
    print(f"    CI for accuracy improvement: [{ci_lower_pct:.4f}%, {ci_upper_pct:.4f}%]")
    print(f"    Excludes 0: {'✅ YES' if not contains_zero else '❌ NO'}")
    print(f"    Result: {'✅ SIGNIFICANT' if not contains_zero else '❌ NOT SIGNIFICANT'}")
    
    # Store results
    results.append({
        'Model': model_name,
        'Improvement (%)': mean_improvement,
        'Improved_Count': n_improved,
        'Worse_Count': n_worse,
        'Same_Count': n_same,
        'Improved_Pct': n_improved/n_samples*100,
        'Worse_Pct': n_worse/n_samples*100,
        'Same_Pct': n_same/n_samples*100,
        'p_ttest': p_ttest,
        'p_wilcox': p_wilcox,
        'p_mcnemar': p_mcnemar,
        'ci_lower (%)': ci_lower_pct,
        'ci_upper (%)': ci_upper_pct,
        'significant_ttest': p_ttest < 0.05,
        'significant_mcnemar': p_mcnemar < 0.05,
        'bootstrap_excludes_zero': not contains_zero
    })

# ==========================================
# 4. SUMMARY TABLE
# ==========================================

print("\n" + "="*80)
print(" 3. SUMMARY TABLE")
print("="*80)

summary_df = pd.DataFrame(results)

# Format p-values with significance indicator
def format_pvalue(p_val):
    if p_val is None:
        return 'N/A'
    if p_val < 0.001:
        return f'{p_val:.6f}***'
    elif p_val < 0.01:
        return f'{p_val:.6f}**'
    elif p_val < 0.05:
        return f'{p_val:.6f}*'
    else:
        return f'{p_val:.6f}'

summary_df['p_ttest_formatted'] = summary_df['p_ttest'].apply(format_pvalue)
summary_df['p_mcnemar_formatted'] = summary_df['p_mcnemar'].apply(format_pvalue)
summary_df['p_wilcox_formatted'] = summary_df['p_wilcox'].apply(lambda x: format_pvalue(x) if x is not None else 'N/A')

print(summary_df[['Model', 'Improvement (%)', 'Improved_Pct', 'Worse_Pct', 'Same_Pct',
                  'p_ttest_formatted', 'p_mcnemar_formatted', 
                  'ci_lower (%)', 'ci_upper (%)']].to_string(index=False))

# ==========================================
# 5. STATISTICAL SIGNIFICANCE SUMMARY
# ==========================================

print("\n" + "="*80)
print(" 4. STATISTICAL SIGNIFICANCE SUMMARY (p < 0.05)")
print("="*80)

for r in results:
    print(f"\n{r['Model']}:")
    print(f"  Improvement: {r['Improvement (%)']:.4f}%")
    print(f"  Improved: {r['Improved_Pct']:.2f}%, Worsened: {r['Worse_Pct']:.2f}%, Same: {r['Same_Pct']:.2f}%")
    print(f"  Paired t-test: p = {r['p_ttest']:.6f} {'✅ SIGNIFICANT' if r['p_ttest'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    print(f"  McNemar's test: p = {r['p_mcnemar']:.6f} {'✅ SIGNIFICANT' if r['p_mcnemar'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    if r['p_wilcox'] is not None:
        print(f"  Wilcoxon: p = {r['p_wilcox']:.6f} {'✅ SIGNIFICANT' if r['p_wilcox'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    print(f"  Bootstrap 95% CI: [{r['ci_lower (%)']:.4f}%, {r['ci_upper (%)']:.4f}%] {'✅ EXCLUDES 0' if r['bootstrap_excludes_zero'] else '❌ CONTAINS 0'}")
    
    # Overall verdict
    if r['significant_ttest'] and r['bootstrap_excludes_zero']:
        print(f"  Overall: ✅ STATISTICALLY SIGNIFICANT")
    else:
        print(f"  Overall: ❌ NOT STATISTICALLY SIGNIFICANT")

# ==========================================
# 6. VISUALIZATION (With per-sample comparison as 4th plot)
# ==========================================

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 14))

# 1. Performance comparison
metrics_df.T[['Accuracy', 'F1']].plot(kind='bar', ax=ax1)
ax1.set_title('Model Performance Comparison')
ax1.set_xlabel('Models')
ax1.set_ylabel('Score')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)
ax1.set_ylim([0, 1])

# 2. Improvement with Confidence Intervals
model_names_short = [r['Model'] for r in results]
improvements = [r['Improvement (%)'] for r in results]
ci_lowers = [r['ci_lower (%)'] for r in results]
ci_uppers = [r['ci_upper (%)'] for r in results]

ax2.errorbar(model_names_short, improvements, 
             yerr=[np.array(improvements)-np.array(ci_lowers), 
                   np.array(ci_uppers)-np.array(improvements)],
             fmt='o', capsize=10, capthick=2, markersize=12, 
             color='navy', ecolor='gray', elinewidth=2)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='No Improvement')
ax2.set_xlabel('Models')
ax2.set_ylabel('Improvement (%)')
ax2.set_title('Accuracy Improvement vs Baseline (95% CI)')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. P-values comparison (formatted for 0.05)
p_values_matrix = []
for r in results:
    p_values_matrix.append([
        r['p_ttest'], 
        r['p_mcnemar'], 
        r['p_wilcox'] if r['p_wilcox'] is not None else np.nan
    ])

p_values_df = pd.DataFrame(p_values_matrix, 
                            index=[r['Model'] for r in results],
                            columns=['Paired T-Test', 'McNemar', 'Wilcoxon'])

# Create formatted p-value labels
def format_pval_plot(p_val):
    if np.isnan(p_val):
        return 'N/A'
    if p_val < 0.001:
        return f'{p_val:.2e}***'
    elif p_val < 0.01:
        return f'{p_val:.4f}**'
    elif p_val < 0.05:
        return f'{p_val:.4f}*'
    elif p_val < 0.1:
        return f'{p_val:.4f}†'
    else:
        return f'{p_val:.4f}'

# Log transform p-values for better visualization
p_values_log = -np.log10(p_values_df + 1e-10)

# Create annotations
annotations = np.array([[format_pval_plot(p_values_df.iloc[i, j]) for j in range(3)] for i in range(len(p_values_df))])

sns.heatmap(p_values_log, annot=annotations, fmt='', cmap='Reds', ax=ax3,
            cbar_kws={'label': '-log10(p-value)'})
ax3.set_title('Statistical Significance Heatmap\n(Higher = More Significant)')
ax3.set_xlabel('Statistical Test')
ax3.set_ylabel('Model')

# Add significance key
ax3.text(0.5, -0.15, '*** p < 0.001, ** p < 0.01, * p < 0.05, † p < 0.1', 
         transform=ax3.transAxes, ha='center', fontsize=10, style='italic')

# 4. Per-sample comparison: Correct, Wrong, Same (Stacked Bar Chart)
model_names_short = [r['Model'] for r in results]
improved_pct = [r['Improved_Pct'] for r in results]
worse_pct = [r['Worse_Pct'] for r in results]
same_pct = [r['Same_Pct'] for r in results]

x = np.arange(len(model_names_short))
width = 0.6

# Create stacked bar chart
p4 = ax4.bar(x, improved_pct, width, label='Improved', color='green', alpha=0.8)
p5 = ax4.bar(x, worse_pct, width, bottom=improved_pct, label='Worsened', color='red', alpha=0.8)
p6 = ax4.bar(x, same_pct, width, bottom=np.array(improved_pct)+np.array(worse_pct), 
             label='Same', color='gray', alpha=0.8)

# Add percentage labels on bars
for i, (imp, wor, sam) in enumerate(zip(improved_pct, worse_pct, same_pct)):
    if imp > 5:
        ax4.text(i, imp/2, f'{imp:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if wor > 5:
        ax4.text(i, imp + wor/2, f'{wor:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if sam > 5:
        ax4.text(i, imp + wor + sam/2, f'{sam:.1f}%', ha='center', va='center', color='black', fontweight='bold', fontsize=9)

ax4.set_xticks(x)
ax4.set_xticklabels(model_names_short)
ax4.set_ylabel('Percentage of Samples (%)')
ax4.set_title('Per-Sample Comparison vs Baseline\n(Improved vs Worsened vs Same)')
ax4.legend(loc='upper right')
ax4.grid(alpha=0.3, axis='y')

# Add horizontal line at 50%
ax4.axhline(y=50, color='black', linestyle='--', alpha=0.5)

# Add a summary annotation
for i, (imp, wor) in enumerate(zip(improved_pct, worse_pct)):
    if imp > wor:
        ax4.text(i, -8, f'Net +{imp-wor:.1f}%', ha='center', va='top', 
                color='green', fontweight='bold', fontsize=10)
    elif wor > imp:
        ax4.text(i, -8, f'Net -{wor-imp:.1f}%', ha='center', va='top', 
                color='red', fontweight='bold', fontsize=10)
    else:
        ax4.text(i, -8, 'Net 0%', ha='center', va='top', 
                color='black', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('V_statistical_significance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# 7. SAVE RESULTS TO CSV
# ==========================================

summary_df.to_csv('V_statistical_tests_results.csv', index=False)
metrics_df.to_csv('V_performance_metrics.csv')

print("\n✅ Results saved to CSV files:")
print("  - V_statistical_tests_results.csv")
print("  - V_performance_metrics.csv")



# Count significant models
significant_models = [r['Model'] for r in results if r['significant_ttest'] and r['bootstrap_excludes_zero']]



# ==========================================
# 8. CONFIDENCE INTERVAL SUMMARY
# ==========================================

print("\n" + "="*80)
print(" 5. PER-SAMPLE COMPARISON SUMMARY")
print("="*80)

per_sample_df = pd.DataFrame({
    'Model': [r['Model'] for r in results],
    'Improved (%)': [r['Improved_Pct'] for r in results],
    'Worsened (%)': [r['Worse_Pct'] for r in results],
    'Same (%)': [r['Same_Pct'] for r in results],
    'Net Effect (%)': [r['Improved_Pct'] - r['Worse_Pct'] for r in results],
    'Significant': ['✅' if (r['significant_ttest'] and r['bootstrap_excludes_zero']) else '❌' for r in results]
})

print(per_sample_df.to_string(index=False))

# ==========================================
# 9. TABLE-READY RESULTS
# ==========================================

print("\n" + "="*80)
print(" TABLE-READY RESULTS (p < 0.05)")
print("="*80)

table_data = []
for r in results:
    table_data.append([
        r['Model'],
        f"{r['Improvement (%)']:.2f}%",
        f"{r['Improved_Pct']:.1f}%",
        f"{r['Worse_Pct']:.1f}%",
        f"{r['Same_Pct']:.1f}%",
        f"{r['p_ttest']:.6f} {'*' if r['p_ttest'] < 0.05 else ''}",
        f"[{r['ci_lower (%)']:.2f}%, {r['ci_upper (%)']:.2f}%]",
        '✅' if (r['significant_ttest'] and r['bootstrap_excludes_zero']) else '❌'
    ])

table_df = pd.DataFrame(table_data, columns=['Model', 'Improvement', 'Improved %', 'Worsened %', 'Same %', 
                                              'p (t-test)', '95% CI', 'Significant'])
print(table_df.to_string(index=False))

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)


In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import ttest_rel, wilcoxon, friedmanchisquare
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# LOAD .NPY FILES 
# ==========================================

# Load true labels (ground truth)
true_labels = np.load('NPY/precise_scores.npy')  # Shape: (8073,)

# Load predictions for all models
base_pred   = np.load('NPY/EN_m_a_tile.npy')      # Shape: (8073,)
model1_pred = np.load('NPY/EN_BN_relu_tile.npy')   # Shape: (8073,)
model2_pred = np.load('NPY/EN_BN_c1_slide.npy')  # Shape: (8073,)
model3_pred = np.load('NPY/EN_m_a_Attn_slide.npy')  # Shape: (8073,)




# Stack all predictions
all_predictions = np.column_stack([
    base_pred, model1_pred, model2_pred, 
    model3_pred
])

model_names = ['EN_m_a', 'EN_BN','EN_BN_c1','EN_m_a_Attn']
n_models = all_predictions.shape[1]
n_samples = len(true_labels)

print("="*80)
print(" STATISTICAL SIGNIFICANCE TESTS")
print(f"   Samples: {n_samples}")
print(f"   Models: {n_models} (En_m_a + 3 improved)")
print("="*80)

# ==========================================
# 1. DESCRIPTIVE STATISTICS
# ==========================================

print("\n 1. DESCRIPTIVE STATISTICS")
print("-" * 60)

metrics_df = pd.DataFrame()

for i, name in enumerate(model_names):
    preds = all_predictions[:, i]
    
    # Overall metrics
    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, average='weighted', zero_division=0)
    rec = recall_score(true_labels, preds, average='weighted', zero_division=0)
    f1 = f1_score(true_labels, preds, average='weighted', zero_division=0)
    
    # Correctness per sample
    correct = (preds == true_labels).astype(int)
    correct_pct = correct.mean() * 100
    
    metrics_df[name] = [acc, prec, rec, f1, correct_pct]
    print(f"  {name:12s}: Acc={acc:.4f}, F1={f1:.4f}, Correct={correct_pct:.2f}%")

metrics_df.index = ['Accuracy', 'Precision', 'Recall', 'F1', 'Correct (%)']
print("\n", metrics_df.round(4))

# ==========================================
# 2. PER-SAMPLE CORRECTNESS MATRIX
# ==========================================

correctness_matrix = np.zeros((n_samples, n_models))
for i in range(n_models):
    correctness_matrix[:, i] = (all_predictions[:, i] == true_labels).astype(int)

# ==========================================
# 3. PAIRED STATISTICAL TESTS (Each Model vs Base)
# ==========================================

print("\n" + "="*80)
print(" 2. PAIRED STATISTICAL TESTS (Each Model vs Base)")
print("="*80)

results = []

for i in range(1, n_models):
    model_name = model_names[i]
    
    # Get per-sample correctness
    base_correct = correctness_matrix[:, 0]
    model_correct = correctness_matrix[:, i]
    
    # Difference per sample
    diff = model_correct - base_correct

    plt.plot(diff)


    # Descriptive stats
    n_improved = np.sum(diff > 0)
    n_worse = np.sum(diff < 0)
    n_same = np.sum(diff == 0)
    mean_improvement = np.mean(diff) * 100
    
    print(f"\n{'='*60}")
    print(f" {model_name} vs Base (n = {n_samples} samples)")
    print(f"{'='*60}")
    print(f"  Improved: {n_improved} ({n_improved/n_samples*100:.2f}%)")
    print(f"  Worsened: {n_worse} ({n_worse/n_samples*100:.2f}%)")
    print(f"  Same:     {n_same} ({n_same/n_samples*100:.2f}%)")
    print(f"  Mean Improvement: {mean_improvement:.4f}%")
    
    # ----- Test 1: Paired T-Test -----
    t_stat, p_ttest = ttest_rel(base_correct, model_correct)
    
    print(f"\n  PAIRED T-TEST:")
    print(f"    t-statistic: {t_stat:.4f}")
    print(f"    p-value:     {p_ttest:.6f}")
    print(f"    p < 0.05:    {'✅ YES' if p_ttest < 0.05 else '❌ NO'}")
    print(f"    Result:      {'✅ SIGNIFICANT' if p_ttest < 0.05 else '❌ NOT SIGNIFICANT'}")
    
    # ----- Test 2: Wilcoxon Signed-Rank Test -----
    non_zero_diff = diff[diff != 0]
    if len(non_zero_diff) > 0:
        w_stat, p_wilcox = wilcoxon(non_zero_diff)
        print(f"\n  WILCOXON SIGNED-RANK TEST:")
        print(f"    W-statistic: {w_stat:.2f}")
        print(f"    p-value:     {p_wilcox:.6f}")
        print(f"    p < 0.05:    {'✅ YES' if p_wilcox < 0.05 else '❌ NO'}")
        print(f"    Result:      {'✅ SIGNIFICANT' if p_wilcox < 0.05 else '❌ NOT SIGNIFICANT'}")
    else:
        p_wilcox = None
        print(f"\n   WILCOXON TEST: All differences are zero")
    
    # ----- Test 3: McNemar's Test -----
    a = np.sum((base_correct == 1) & (model_correct == 1))
    b = np.sum((base_correct == 1) & (model_correct == 0))
    c = np.sum((base_correct == 0) & (model_correct == 1))
    d = np.sum((base_correct == 0) & (model_correct == 0))
    
    contingency_table = np.array([[a, b], [c, d]])
    mcnemar_result = mcnemar(contingency_table, exact=False, correction=True)
    p_mcnemar = mcnemar_result.pvalue
    
    print(f"\n  McNEMAR'S TEST:")
    print(f"    Contingency table:")
    print(f"                | Base Correct | Base Wrong")
    print(f"    {model_name} Correct |     {a:5d}     |    {c:5d}")
    print(f"    {model_name} Wrong   |     {b:5d}     |    {d:5d}")
    print(f"    p-value: {p_mcnemar:.6f}")
    print(f"    p < 0.05: {'✅ YES' if p_mcnemar < 0.05 else '❌ NO'}")
    print(f"    Result:  {'✅ SIGNIFICANT' if p_mcnemar < 0.05 else '❌ NOT SIGNIFICANT'}")
    
    # ----- Test 4: Bootstrap Confidence Interval -----
    n_bootstrap = 4000
    bootstrap_diffs = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n_samples, size=n_samples, replace=True)
        boot_base = base_correct[idx]
        boot_model = model_correct[idx]
        bootstrap_diffs.append(np.mean(boot_model) - np.mean(boot_base))
    
    ci_lower = np.percentile(bootstrap_diffs, 2.5)
    ci_upper = np.percentile(bootstrap_diffs, 97.5)
    ci_lower_pct = ci_lower * 100
    ci_upper_pct = ci_upper * 100
    contains_zero = ci_lower <= 0 <= ci_upper
    
    print(f"\n  BOOTSTRAP 95% CONFIDENCE INTERVAL:")
    print(f"    CI for accuracy improvement: [{ci_lower_pct:.4f}%, {ci_upper_pct:.4f}%]")
    print(f"    Excludes 0: {'✅ YES' if not contains_zero else '❌ NO'}")
    print(f"    Result: {'✅ SIGNIFICANT' if not contains_zero else '❌ NOT SIGNIFICANT'}")
    
    # Store results
    results.append({
        'Model': model_name,
        'Improvement (%)': mean_improvement,
        'Improved_Count': n_improved,
        'Worse_Count': n_worse,
        'Same_Count': n_same,
        'Improved_Pct': n_improved/n_samples*100,
        'Worse_Pct': n_worse/n_samples*100,
        'Same_Pct': n_same/n_samples*100,
        'p_ttest': p_ttest,
        'p_wilcox': p_wilcox,
        'p_mcnemar': p_mcnemar,
        'ci_lower (%)': ci_lower_pct,
        'ci_upper (%)': ci_upper_pct,
        'significant_ttest': p_ttest < 0.05,
        'significant_mcnemar': p_mcnemar < 0.05,
        'bootstrap_excludes_zero': not contains_zero
    })

# ==========================================
# 4. SUMMARY TABLE
# ==========================================

print("\n" + "="*80)
print("📋 3. SUMMARY TABLE")
print("="*80)

summary_df = pd.DataFrame(results)

# Format p-values with significance indicator
def format_pvalue(p_val):
    if p_val is None:
        return 'N/A'
    if p_val < 0.001:
        return f'{p_val:.6f}***'
    elif p_val < 0.01:
        return f'{p_val:.6f}**'
    elif p_val < 0.05:
        return f'{p_val:.6f}*'
    else:
        return f'{p_val:.6f}'

summary_df['p_ttest_formatted'] = summary_df['p_ttest'].apply(format_pvalue)
summary_df['p_mcnemar_formatted'] = summary_df['p_mcnemar'].apply(format_pvalue)
summary_df['p_wilcox_formatted'] = summary_df['p_wilcox'].apply(lambda x: format_pvalue(x) if x is not None else 'N/A')

print(summary_df[['Model', 'Improvement (%)', 'Improved_Pct', 'Worse_Pct', 'Same_Pct',
                  'p_ttest_formatted', 'p_mcnemar_formatted', 
                  'ci_lower (%)', 'ci_upper (%)']].to_string(index=False))

# ==========================================
# 5. STATISTICAL SIGNIFICANCE SUMMARY
# ==========================================

print("\n" + "="*80)
print(" 4. STATISTICAL SIGNIFICANCE SUMMARY (p < 0.05)")
print("="*80)

for r in results:
    print(f"\n{r['Model']}:")
    print(f"  Improvement: {r['Improvement (%)']:.4f}%")
    print(f"  Improved: {r['Improved_Pct']:.2f}%, Worsened: {r['Worse_Pct']:.2f}%, Same: {r['Same_Pct']:.2f}%")
    print(f"  Paired t-test: p = {r['p_ttest']:.6f} {'✅ SIGNIFICANT' if r['p_ttest'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    print(f"  McNemar's test: p = {r['p_mcnemar']:.6f} {'✅ SIGNIFICANT' if r['p_mcnemar'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    if r['p_wilcox'] is not None:
        print(f"  Wilcoxon: p = {r['p_wilcox']:.6f} {'✅ SIGNIFICANT' if r['p_wilcox'] < 0.05 else '❌ NOT SIGNIFICANT'}")
    print(f"  Bootstrap 95% CI: [{r['ci_lower (%)']:.4f}%, {r['ci_upper (%)']:.4f}%] {'✅ EXCLUDES 0' if r['bootstrap_excludes_zero'] else '❌ CONTAINS 0'}")
    
    # Overall verdict
    if r['significant_ttest'] and r['bootstrap_excludes_zero']:
        print(f"  Overall: ✅ STATISTICALLY SIGNIFICANT")
    else:
        print(f"  Overall: ❌ NOT STATISTICALLY SIGNIFICANT")

# ==========================================
# 6. VISUALIZATION (With per-sample comparison as 4th plot)
# ==========================================

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 14))

# 1. Performance comparison
metrics_df.T[['Accuracy', 'F1']].plot(kind='bar', ax=ax1)
ax1.set_title('Model Performance Comparison')
ax1.set_xlabel('Models')
ax1.set_ylabel('Score')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)
ax1.set_ylim([0, 1])

# 2. Improvement with Confidence Intervals
model_names_short = [r['Model'] for r in results]
improvements = [r['Improvement (%)'] for r in results]
ci_lowers = [r['ci_lower (%)'] for r in results]
ci_uppers = [r['ci_upper (%)'] for r in results]

ax2.errorbar(model_names_short, improvements, 
             yerr=[np.array(improvements)-np.array(ci_lowers), 
                   np.array(ci_uppers)-np.array(improvements)],
             fmt='o', capsize=10, capthick=2, markersize=12, 
             color='navy', ecolor='gray', elinewidth=2)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='No Improvement')
ax2.set_xlabel('Models')
ax2.set_ylabel('Improvement (%)')
ax2.set_title('Accuracy Improvement vs Baseline (95% CI)')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. P-values comparison (formatted for 0.05)
p_values_matrix = []
for r in results:
    p_values_matrix.append([
        r['p_ttest'], 
        r['p_mcnemar'], 
        r['p_wilcox'] if r['p_wilcox'] is not None else np.nan
    ])

p_values_df = pd.DataFrame(p_values_matrix, 
                            index=[r['Model'] for r in results],
                            columns=['Paired T-Test', 'McNemar', 'Wilcoxon'])

# Create formatted p-value labels
def format_pval_plot(p_val):
    if np.isnan(p_val):
        return 'N/A'
    if p_val < 0.001:
        return f'{p_val:.2e}***'
    elif p_val < 0.01:
        return f'{p_val:.4f}**'
    elif p_val < 0.05:
        return f'{p_val:.4f}*'
    elif p_val < 0.1:
        return f'{p_val:.4f}†'
    else:
        return f'{p_val:.4f}'

# Log transform p-values for better visualization
p_values_log = -np.log10(p_values_df + 1e-10)

# Create annotations
annotations = np.array([[format_pval_plot(p_values_df.iloc[i, j]) for j in range(3)] for i in range(len(p_values_df))])

sns.heatmap(p_values_log, annot=annotations, fmt='', cmap='Reds', ax=ax3,
            cbar_kws={'label': '-log10(p-value)'})
ax3.set_title('Statistical Significance Heatmap\n(Higher = More Significant)')
ax3.set_xlabel('Statistical Test')
ax3.set_ylabel('Model')

# Add significance key
ax3.text(0.5, -0.15, '*** p < 0.001, ** p < 0.01, * p < 0.05, † p < 0.1', 
         transform=ax3.transAxes, ha='center', fontsize=10, style='italic')

# 4. Per-sample comparison: Correct, Wrong, Same (Stacked Bar Chart)
model_names_short = [r['Model'] for r in results]
improved_pct = [r['Improved_Pct'] for r in results]
worse_pct = [r['Worse_Pct'] for r in results]
same_pct = [r['Same_Pct'] for r in results]

x = np.arange(len(model_names_short))
width = 0.6

# Create stacked bar chart
p4 = ax4.bar(x, improved_pct, width, label='Improved', color='green', alpha=0.8)
p5 = ax4.bar(x, worse_pct, width, bottom=improved_pct, label='Worsened', color='red', alpha=0.8)
p6 = ax4.bar(x, same_pct, width, bottom=np.array(improved_pct)+np.array(worse_pct), 
             label='Same', color='gray', alpha=0.8)

# Add percentage labels on bars
for i, (imp, wor, sam) in enumerate(zip(improved_pct, worse_pct, same_pct)):
    if imp > 5:
        ax4.text(i, imp/2, f'{imp:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if wor > 5:
        ax4.text(i, imp + wor/2, f'{wor:.1f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if sam > 5:
        ax4.text(i, imp + wor + sam/2, f'{sam:.1f}%', ha='center', va='center', color='black', fontweight='bold', fontsize=9)

ax4.set_xticks(x)
ax4.set_xticklabels(model_names_short)
ax4.set_ylabel('Percentage of Samples (%)')
ax4.set_title('Per-Sample Comparison vs Baseline\n(Improved vs Worsened vs Same)')
ax4.legend(loc='upper right')
ax4.grid(alpha=0.3, axis='y')

# Add horizontal line at 50%
ax4.axhline(y=50, color='black', linestyle='--', alpha=0.5)

# Add a summary annotation
for i, (imp, wor) in enumerate(zip(improved_pct, worse_pct)):
    if imp > wor:
        ax4.text(i, -8, f'Net +{imp-wor:.1f}%', ha='center', va='top', 
                color='green', fontweight='bold', fontsize=10)
    elif wor > imp:
        ax4.text(i, -8, f'Net -{wor-imp:.1f}%', ha='center', va='top', 
                color='red', fontweight='bold', fontsize=10)
    else:
        ax4.text(i, -8, 'Net 0%', ha='center', va='top', 
                color='black', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('statistical_significance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# 7. SAVE RESULTS TO CSV
# ==========================================

summary_df.to_csv('E_statistical_tests_results.csv', index=False)
metrics_df.to_csv('E_performance_metrics.csv')

print("\n✅ Results saved to CSV files:")
print("  - statistical_tests_results.csv")
print("  - performance_metrics.csv")


# ==========================================
# 8. CONFIDENCE INTERVAL SUMMARY
# ==========================================

print("\n" + "="*80)
print("📊 5. PER-SAMPLE COMPARISON SUMMARY")
print("="*80)

per_sample_df = pd.DataFrame({
    'Model': [r['Model'] for r in results],
    'Improved (%)': [r['Improved_Pct'] for r in results],
    'Worsened (%)': [r['Worse_Pct'] for r in results],
    'Same (%)': [r['Same_Pct'] for r in results],
    'Net Effect (%)': [r['Improved_Pct'] - r['Worse_Pct'] for r in results],
    'Significant': ['✅' if (r['significant_ttest'] and r['bootstrap_excludes_zero']) else '❌' for r in results]
})

print(per_sample_df.to_string(index=False))

# ==========================================
# 9. TABLE-READY RESULTS
# ==========================================

print("\n" + "="*80)
print("📋 TABLE-READY RESULTS (p < 0.05)")
print("="*80)

table_data = []
for r in results:
    table_data.append([
        r['Model'],
        f"{r['Improvement (%)']:.2f}%",
        f"{r['Improved_Pct']:.1f}%",
        f"{r['Worse_Pct']:.1f}%",
        f"{r['Same_Pct']:.1f}%",
        f"{r['p_ttest']:.6f} {'*' if r['p_ttest'] < 0.05 else ''}",
        f"[{r['ci_lower (%)']:.2f}%, {r['ci_upper (%)']:.2f}%]",
        '✅' if (r['significant_ttest'] and r['bootstrap_excludes_zero']) else '❌'
    ])

table_df = pd.DataFrame(table_data, columns=['Model', 'Improvement', 'Improved %', 'Worsened %', 'Same %', 
                                              'p (t-test)', '95% CI', 'Significant'])
print(table_df.to_string(index=False))

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)
